# Course 2 — Data Manipulation (Pandas)

Live-coding notebook that mirrors the slide deck.

**Sections**
1. DataFrames & Loading (0:00–0:30)
2. Selection & Filtering (0:30–1:00)
3. Cleaning & Grouping (1:00–1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
import pandas as pd
import numpy as np
from shared.data_utils import load_dataset
pd.__version__


## 1. DataFrames & Loading

In [ ]:
df = load_dataset('penguins')
df.head()


In [ ]:
df.info()
df.describe(include='all')


### Index vs columns

In [ ]:
print('index:', df.index)
print('cols :', df.columns.tolist())
# Set a more meaningful index
df2 = df.reset_index().rename(columns={'index': 'penguin_id'}).set_index('penguin_id')
df2.head()


## 2. Selection & Filtering

### `.loc` (label) vs `.iloc` (position)

In [ ]:
df.loc[0:3, ['species', 'island', 'bill_length_mm']]


In [ ]:
df.iloc[0:3, 0:3]


### Boolean indexing — the workhorse

In [ ]:
adelie_long = df[(df['species'] == 'Adelie') & (df['bill_length_mm'] > 40)]
adelie_long.head()


In [ ]:
# Cleaner syntax for complex filters
df.query('species == "Adelie" and bill_length_mm > 40').head()


## 3. Cleaning & Grouping

### Missing values

In [ ]:
df.isna().sum()


In [ ]:
clean = df.dropna()
print('before:', len(df), '  after dropna:', len(clean))


Or impute instead of dropping:

In [ ]:
filled = df.copy()
filled['bill_length_mm'] = filled['bill_length_mm'].fillna(filled['bill_length_mm'].median())
filled.isna().sum()


### groupby + agg

In [ ]:
clean.groupby('species')[['bill_length_mm', 'body_mass_g']].mean().round(2)


In [ ]:
clean.groupby(['species', 'sex']).agg(
    n=('bill_length_mm', 'size'),
    mass_mean=('body_mass_g', 'mean'),
    mass_std=('body_mass_g', 'std'),
).round(1)


### Feature engineering — derive a new column

In [ ]:
clean = clean.assign(
    bill_ratio=lambda d: d['bill_length_mm'] / d['bill_depth_mm'],
    mass_kg=lambda d: d['body_mass_g'] / 1000,
)
clean[['species', 'bill_ratio', 'mass_kg']].head()


### Recap
* Treat a DataFrame like a typed, indexed table.
* `.loc` for labels, `.iloc` for positions, boolean masks for filtering.
* Always inspect NaNs before training anything on the data.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

# Exercise 1 — DataFrame I/O & Inspection

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd


**Task 1.** Load the `tips` dataset. Show the first 5 rows and the dtypes.

In [ ]:
# your code here


**Task 2.** How many rows and columns does it have? How many numeric columns?

In [ ]:
# your code here


**Task 3.** Print a 5-number summary (`describe()`) for the numeric columns only.

In [ ]:
# your code here


**Task 4.** Set the index to a new column `bill_id` (just a 0-based range).

In [ ]:
# your code here


### Exercise 1 — Solution

# Exercise 1 — Solutions

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd
df = load_dataset('tips')


**Task 1.**

In [ ]:
print(df.head())
print(df.dtypes)


**Task 2.**

In [ ]:
print('shape:', df.shape)
print('numeric cols:', df.select_dtypes('number').columns.tolist())


**Task 3.**

In [ ]:
df.describe()


**Task 4.**

In [ ]:
df = df.assign(bill_id=range(len(df))).set_index('bill_id')
df.head()


---

## Exercise 2

# Exercise 2 — Selection & Filtering

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
df = load_dataset('tips')


**Task 1.** Select rows 10–19 (inclusive) and only the columns `total_bill`, `tip`, `day`. Use `.loc`.

In [ ]:
# your code here


**Task 2.** Find all dinners where the tip was at least 20% of the bill.

In [ ]:
# your code here


**Task 3.** Same as Task 2 but use `df.query(...)`.

In [ ]:
# your code here


**Task 4.** What is the average tip on weekends (Sat/Sun) vs weekdays?

In [ ]:
# your code here


### Exercise 2 — Solution

# Exercise 2 — Solutions

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
df = load_dataset('tips')


**Task 1.**

In [ ]:
df.loc[10:19, ['total_bill', 'tip', 'day']]


**Task 2.**

In [ ]:
mask = (df['time'] == 'Dinner') & (df['tip'] / df['total_bill'] >= 0.20)
df[mask].head()


**Task 3.**

In [ ]:
df.query('time == "Dinner" and tip / total_bill >= 0.20').head()


**Task 4.**

In [ ]:
(df.assign(weekend=df['day'].isin(['Sat', 'Sun']))
   .groupby('weekend')['tip'].mean().round(2))


---

## Exercise 3

# Exercise 3 — Cleaning & Grouping

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
df = load_dataset('penguins')


**Task 1.** Report the number of NaNs per column.

In [ ]:
# your code here


**Task 2.** Drop every row that has at least one NaN. How many rows did you lose?

In [ ]:
# your code here


**Task 3.** On the cleaned data: per-species mean and std of `body_mass_g`. Round to 1 decimal.

In [ ]:
# your code here


**Task 4.** Add a column `mass_kg` (= body_mass_g / 1000) and a column `bill_ratio` (= bill_length / bill_depth). Sort by `bill_ratio` descending and show the top 5.

In [ ]:
# your code here


### Exercise 3 — Solution

# Exercise 3 — Solutions

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
df = load_dataset('penguins')


**Task 1.**

In [ ]:
df.isna().sum()


**Task 2.**

In [ ]:
clean = df.dropna()
print('lost', len(df) - len(clean), 'rows')


**Task 3.**

In [ ]:
clean.groupby('species')['body_mass_g'].agg(['mean', 'std']).round(1)


**Task 4.**

In [ ]:
clean = clean.assign(
    mass_kg=clean['body_mass_g'] / 1000,
    bill_ratio=clean['bill_length_mm'] / clean['bill_depth_mm'],
)
clean.sort_values('bill_ratio', ascending=False).head()
